# 🔬 OptoBind-AI — Pipeline de Design De Novo de Biosenseurs Optiques

Ce notebook exécute **OptoBind-AI** dans un environnement propre **Python 3.10 natif + PyTorch CUDA** sur un **GPU NVIDIA gratuit dans Google Colab**.

---

### Étape 1 : Vérification du GPU NVIDIA T4
*(Assurez-vous que le GPU est actif dans Exécution > Modifier le type d'exécution > T4 GPU)*

In [ ]:
!nvidia-smi

### Étape 2 : Cloner le projet Optobind-AI

In [ ]:
%cd /content
!rm -rf Optobind-AI
!git clone https://github.com/Lorenzo-colao/Optobind-AI.git
%cd /content/Optobind-AI

!git clone --recurse-submodules https://github.com/baker-laboratory/rf_diffusion_all_atom.git RFdiffusion 2>/dev/null || true
!git clone https://github.com/dauparas/ProteinMPNN.git 2>/dev/null || true

print("\n✓ Dépôts sources prêts !")

### Étape 3 : Installation de l'environnement Python 3.10 natif

In [ ]:
%%time
# 1. Installer Python 3.10 via apt Ubuntu
!add-apt-repository -y ppa:deadsnakes/ppa > /dev/null 2>&1
!apt-get update -qq > /dev/null 2>&1 || true
!apt-get install -y -qq python3.10 python3.10-venv python3.10-distutils > /dev/null 2>&1

# 2. Créer l'environnement virtuel Python 3.10 dédié si absent
![ -d /content/optobind_env ] || python3.10 -m venv /content/optobind_env

# 3. Installer PyTorch avec CUDA 11.8 officiel pour Python 3.10
!/content/optobind_env/bin/pip install --quiet --upgrade pip
!/content/optobind_env/bin/pip install --quiet "torch==2.5.1+cu118" "torchvision==0.20.1+cu118" "torchaudio==2.5.1+cu118" --index-url https://download.pytorch.org/whl/cu118
!/content/optobind_env/bin/pip install --quiet "torchdata==0.7.1"

# 4. Installer DGL compatible CUDA 11.8 pour Python 3.10
!/content/optobind_env/bin/pip install --quiet "dgl==1.1.3" -f https://data.dgl.ai/wheels/cu118/repo.html || /content/optobind_env/bin/pip install --quiet "dgl==1.1.3" -f https://data.dgl.ai/wheels/repo.html || /content/optobind_env/bin/pip install --quiet "dgl==1.1.3"

# 5. Installer les bibliothèques du pipeline
!/content/optobind_env/bin/pip install --quiet biopython rdkit openpyxl pandas pyyaml tqdm hydra-core omegaconf pynvml decorator "e3nn==0.3.3" "chai-lab>=0.6.0" fair-esm icecream opt_einsum scipy openbabel-wheel assertpy deepdiff
!/content/optobind_env/bin/pip install --quiet git+https://github.com/NVIDIA/dllogger#egg=dllogger

# 6. Enregistrer RFdiffusion et SE3Transformer dans l'environnement virtuel
# rf_diffusion_all_atom intègre rf2aa dans lib/rf2aa
!/content/optobind_env/bin/pip install --quiet -e RFdiffusion/lib/rf2aa 2>/dev/null || true

# 7. Télécharger les poids s'ils ne sont pas présents
!mkdir -p weights
!wget -c -P weights/ http://files.ipd.uw.edu/pub/RF-All-Atom/weights/RFDiffusionAA_paper_weights.pt
!wget -c -O weights/ProteinMPNN_v_48_020.pt https://raw.githubusercontent.com/dauparas/ProteinMPNN/main/vanilla_model_weights/v_48_020.pt

# 8. Appliquer les correctifs automatiques pour RFdiffusion All-Atom
!/content/optobind_env/bin/python -c "from optobind_ai.utils.patcher import patch_all_rfdiffusion; patch_all_rfdiffusion(); print('✓ RFdiffusion All-Atom patché avec succès !')"

print("\n🎉 ENVIRONNEMENT PYTHON 3.10 DÉDIÉ PRÊT !")

### Étape 4 : Lancer le pipeline OptoBind-AI sur GPU !

In [ ]:
!/content/optobind_env/bin/python orchestrator.py \
  --target inputs/targets/zn2+_site.pdb \
  --ligand inputs/ligands/DFHBI-1T.sdf

### Étape 5 : Télécharger votre rapport Excel de validation

In [ ]:
import glob
import subprocess
from google.colab import files

# Création d'une archive zip de tous les résultats (PDBs, séquences, rapports)
subprocess.run('zip -q -r optobind_results.zip outputs/', shell=True)
if glob.glob('optobind_results.zip'):
    print("📦 Téléchargement de l'archive complète des résultats (optobind_results.zip)...")
    files.download('optobind_results.zip')

# Recherche et téléchargement du rapport Excel dédié
reports = glob.glob('outputs/**/reports/report.xlsx', recursive=True)
if reports:
    print(f"📊 Téléchargement du rapport Excel : {reports[-1]}")
    files.download(reports[-1])
else:
    print("Aucun rapport Excel trouvé.")